# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimakhaliq/flyrank-ml-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: word_count (linked to the "thin content" flag concept)
Signal 2: impressions_90d (linked to the "quick-win" volume flag)

In [1]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/fatimakhaliq/flyrank-ml-work/main/data/raw/content_refresh_anonymized.csv")

# Signal 1: word_count vs trend_direction
wc_bucket = pd.cut(df["word_count"], bins=[0,500,1500,5000])
table1 = df.groupby(wc_bucket)["trend_direction"].value_counts()
print("Signal 1 - Word Count buckets:")
print(table1)
print("n =", len(df))

# Signal 2: impressions_90d vs trend_direction
imp_bucket = pd.cut(df["impressions_90d"], bins=[0,100,500,5000,100000])
table2 = df.groupby(imp_bucket)["trend_direction"].value_counts()
print("\nSignal 2 - Impressions buckets:")
print(table2)
print("n =", len(df))


Signal 1 - Word Count buckets:
word_count    trend_direction
(0, 500]      new                   2
              down                  1
              flat                  0
              stable                0
              up                    0
(500, 1500]   down               1449
              new                 870
              up                  325
              stable              300
              flat                281
(1500, 5000]  down               9512
              stable             2819
              up                 2196
              new                1212
              flat                682
Name: count, dtype: int64
n = 30000

Signal 2 - Impressions buckets:
impressions_90d  trend_direction
(0, 100]         down               3116
                 new                2026
                 flat               1116
                 up                 1015
                 stable              733
(100, 500]       down               3190
                 up  

/tmp/ipykernel_16064/1707005250.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table1 = df.groupby(wc_bucket)["trend_direction"].value_counts()
/tmp/ipykernel_16064/1707005250.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table2 = df.groupby(imp_bucket)["trend_direction"].value_counts()


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Verdict Signal 1 (word_count): MIXED — no clear pattern across buckets, similar distribution of down/up/stable in each.
Verdict Signal 2 (impressions_90d): CONFIRMED — pages with higher impressions show more variation in trend, suggesting volume matters for review priority.

In [2]:
# Section 2: Encode one rule (score, reason code, action label)

import pandas as pd
import os

df = pd.read_csv("https://raw.githubusercontent.com/fatimakhaliq/flyrank-ml-work/main/data/raw/content_refresh_anonymized.csv")

# Build baseline score
df["baseline_score"] = (
    (df["impressions_90d"] / df["impressions_90d"].max()) * 0.5 +
    (df["trend_direction"] == "down").astype(int) * 0.5
)

# One reason code
df["reason_code"] = "declining_with_demand"

# One action label
df["action"] = "review_for_refresh"

# Sort by score, highest first
df_sorted = df.sort_values("baseline_score", ascending=False)

# Make sure the output folder exists
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked queue as CSV
df_sorted[["content_id", "baseline_score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)

print("CSV saved successfully!")
print("\nTop 10 rows:")
print(df_sorted[["content_id", "baseline_score", "reason_code", "action"]].head(10))

CSV saved successfully!

Top 10 rows:
                 content_id  baseline_score            reason_code  \
6653   content_5fe46e04994d        1.000000  declining_with_demand   
26844  content_8c19996aa890        0.991827  declining_with_demand   
21819  content_4c36c775b818        0.947257  declining_with_demand   
29879  content_1a9e894be2e2        0.901939  declining_with_demand   
13537  content_2c2606c5d176        0.835512  declining_with_demand   
26531  content_cb112fce36be        0.799306  declining_with_demand   
21565  content_9532f197bbc8        0.798612  declining_with_demand   
27478  content_008fb02c46cb        0.728700  declining_with_demand   
23767  content_813e88069237        0.725569  declining_with_demand   
26304  content_ff94c9b6b411        0.720745  declining_with_demand   

                   action  
6653   review_for_refresh  
26844  review_for_refresh  
21819  review_for_refresh  
29879  review_for_refresh  
13537  review_for_refresh  
26531  review_for_refre

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top20 = df_sorted.head(20).reset_index(drop=True)
for i, row in top20.iterrows():
    print(f"Row {i+1}")
    print(f"  Action: {row['action']}")
    print(f"  Score: {row['baseline_score']:.2f} | Reason code: {row['reason_code']}")
    print("-" * 40)

Row 1
  Action: review_for_refresh
  Score: 1.00 | Reason code: declining_with_demand
----------------------------------------
Row 2
  Action: review_for_refresh
  Score: 0.99 | Reason code: declining_with_demand
----------------------------------------
Row 3
  Action: review_for_refresh
  Score: 0.95 | Reason code: declining_with_demand
----------------------------------------
Row 4
  Action: review_for_refresh
  Score: 0.90 | Reason code: declining_with_demand
----------------------------------------
Row 5
  Action: review_for_refresh
  Score: 0.84 | Reason code: declining_with_demand
----------------------------------------
Row 6
  Action: review_for_refresh
  Score: 0.80 | Reason code: declining_with_demand
----------------------------------------
Row 7
  Action: review_for_refresh
  Score: 0.80 | Reason code: declining_with_demand
----------------------------------------
Row 8
  Action: review_for_refresh
  Score: 0.73 | Reason code: declining_with_demand
-------------------------

Row 1 — Why: Score 1.00 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 2 — Why: Score 0.99 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 3 — Why: Score 0.95 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 4 — Why: Score 0.90 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 5 — Why: Score 0.84 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 6 — Why: Score 0.80 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.
Note: tied with Row 7 at the same score — worth comparing both before deciding which to prioritize.

Row 7 — Why: Score 0.80 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.
Note: tied with Row 6 — same score, so priority between them needs manual review.

Row 8 — Why: Score 0.73 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 9 — Why: Score 0.73 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 10 — Why: Score 0.72 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 11 — Why: Score 0.71 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 12 — Why: Score 0.71 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 13 — Why: Score 0.70 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 14 — Why: Score 0.70 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 15 — Why: Score 0.69 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 16 — Why: Score 0.69 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 17 — Why: Score 0.68 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 18 — Why: Score 0.68 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.

Row 19 — Why: Score 0.67 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.
Note: tied with Row 20 — very close scores mean small data changes could reorder these.

Row 20 — Why: Score 0.67 because impressions were high and the trend was declining.
What would make it wrong: If this drop is seasonal/temporary rather than a permanent decline.
Note: tied with Row 19 — same reasoning applies here.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: All 20 rows share the same action (review_for_refresh) and reason code (declining_with_demand) — only the score differs. This means my rule is only capturing one pattern and may be missing content that needs attention for other reasons (e.g., low demand but stable, or high demand with a sudden single-week dip).

Rows 6/7 and 19/20 have tied or near-tied scores, which makes their relative ranking unstable — a small data change could flip their order.

Leakage check: The score is built only from impressions_90d and trend_direction, both of which are historical/already-observed data. No future-window data (e.g., next quarter's traffic) or client-identifying information (names, URLs) was used in the score, reason code, or action.

In [4]:
# Section 4: Weak picks + leakage check
print("Columns used in scoring:", ["impressions_90d", "trend_direction"])
print("\nUnique reason codes in top 20:", top20['reason_code'].unique())
print("Unique actions in top 20:", top20['action'].unique())

Columns used in scoring: ['impressions_90d', 'trend_direction']

Unique reason codes in top 20: ['declining_with_demand']
Unique actions in top 20: ['review_for_refresh']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.